In [11]:
import numpy as np
import tensorflow as tf
import datetime
import os

# 1. Load data dari seluruh file .npz
train_data = np.load('train_mfcc.npz')
dev_data = np.load('dev_mfcc.npz')
test_data = np.load('test_mfcc.npz')
metadata = np.load('metadata.npz')
validated_data = np.load('validated_mfcc.npz')

print("Data berhasil dimuat!")

Data berhasil dimuat!


In [12]:
# 2. Ekstrak fitur (X) dan label (y) menggunakan key yang sesuai
X_train, y_train = train_data['features'], train_data['labels']
X_dev, y_dev = dev_data['features'], dev_data['labels']
X_test, y_test = test_data['features'], test_data['labels']
X_val_back, y_val_back = validated_data['features'], validated_data['labels']

num_classes = len(np.unique(y_train))
input_shape = (X_train.shape[1], X_train.shape[2])

print("=== VERIFIKASI DIMENSI DATA ===")
print(f"X_train shape : {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_dev shape   : {X_dev.shape} | y_dev shape  : {y_dev.shape}")
print(f"X_test shape  : {X_test.shape} | y_test shape : {y_test.shape}")
print(f"Jumlah Kelas Intent: {num_classes}")

# 3. Konversi ke tf.data.Dataset Pipeline untuk efisiensi training
BATCH_SIZE = 32
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1024).batch(BATCH_SIZE)
dev_dataset = tf.data.Dataset.from_tensor_slices((X_dev, y_dev)).batch(BATCH_SIZE)
test_dataset = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE)

=== VERIFIKASI DIMENSI DATA ===
X_train shape : (22425, 128, 87) | y_train shape: (22425,)
X_dev shape   : (3469, 128, 87) | y_dev shape  : (3469,)
X_test shape  : (3693, 128, 87) | y_test shape : (3693,)
Jumlah Kelas Intent: 5


In [13]:
# Setup Direktori Log untuk TensorBoard Monitoring
log_dir = os.path.join("logs", "gradient_tape", datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
train_summary_writer = tf.summary.create_file_writer(os.path.join(log_dir, "train"))
val_summary_writer = tf.summary.create_file_writer(os.path.join(log_dir, "val"))

print(f"TensorBoard logs akan disimpan di: {log_dir}")
print("Untuk memantau, jalankan perintah ini di terminal: tensorboard --logdir logs/gradient_tape/")

TensorBoard logs akan disimpan di: logs\gradient_tape\20260518-141534
Untuk memantau, jalankan perintah ini di terminal: tensorboard --logdir logs/gradient_tape/


In [14]:
@tf.keras.utils.register_keras_serializable(package="Custom Layers")
class KustomAttention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(KustomAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)
        super(KustomAttention, self).build(input_shape)

    def call(self, x):
        e = tf.tanh(tf.matmul(x, self.W) + self.b)
        alignment_scores = tf.nn.softmax(e, axis=1)
        context_vector = x * alignment_scores
        return tf.reduce_sum(context_vector, axis=1)

print("Custom Attention Layer berhasil didefinisikan dan diregistrasi!")

Custom Attention Layer berhasil didefinisikan dan diregistrasi!


In [15]:
# Membangun Model via Functional API: Input -> CNN -> BiLSTM -> Attention -> Dense -> Softmax
inputs = tf.keras.Input(shape=input_shape, name="Input_MFCC")

# 1. CNN Component
x = tf.keras.layers.Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
x = tf.keras.layers.MaxPooling1D(pool_size=2)(x)
x = tf.keras.layers.Dropout(0.3)(x)

# 2. Bidirectional LSTM Component
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True))(x)

# 3. Custom Attention Component
x = KustomAttention()(x)

# 4. Dense & Output Component
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name="Intent_Output")(x)

# Inisialisasi Model
model = tf.keras.Model(inputs=inputs, outputs=outputs, name="Model_Intent_Audio")
model.summary()

Model: "Model_Intent_Audio"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ Input_MFCC (InputLayer)         │ (None, 128, 87)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 128, 64)        │        16,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64, 128)        │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ kustom_attention_1              │ (None, 128)            │           192 │
│ (KustomAttention)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Intent_Output (Dense)           │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 91,589 (357.77 KB)

 Trainable params: 91,589 (357.77 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
@tf.keras.utils.register_keras_serializable(package="Custom Layers")
class KustomAttention(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(KustomAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                 initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(name="att_bias", shape=(input_shape[1], 1),
                                 initializer="zeros", trainable=True)
        super(KustomAttention, self).build(input_shape)

    def call(self, x):
        e = tf.tanh(tf.matmul(x, self.W) + self.b)
        alignment_scores = tf.nn.softmax(e, axis=1)
        context_vector = x * alignment_scores
        return tf.reduce_sum(context_vector, axis=1)

# Membangun Model via Functional API
inputs = tf.keras.Input(shape=input_shape, name="Input_MFCC")
x = tf.keras.layers.Conv1D(64, kernel_size=3, activation='relu', padding='same')(inputs)
x = tf.keras.layers.MaxPooling1D(pool_size=2)(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True))(x)
x = KustomAttention()(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name="Intent_Output")(x)

model = tf.keras.Model(inputs=inputs, outputs=outputs, name="Model_Intent_Audio")

In [16]:
# 1. Definisikan Loss Function dan Optimizer
loss_object = tf.keras.losses.SparseCategoricalCrossentropy()
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

# 2. Definisikan Metrics Tracker
train_loss_metric = tf.keras.metrics.Mean(name='train_loss')
train_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
train_mae_metric = tf.keras.metrics.MeanAbsoluteError(name='train_mae')

val_loss_metric = tf.keras.metrics.Mean(name='val_loss')
val_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
val_mae_metric = tf.keras.metrics.MeanAbsoluteError(name='val_mae')

# 3. Fungsi Train Step (Per Batch)
@tf.function
def train_step(features, labels):
    with tf.GradientTape() as tape:
        predictions = model(features, training=True)
        loss = loss_object(labels, predictions)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    
    train_loss_metric(loss)
    train_acc_metric(labels, predictions)
    train_mae_metric(tf.one_hot(labels, depth=num_classes), predictions)

# 4. Fungsi Validation Step (Per Batch)
@tf.function
def val_step(features, labels):
    predictions = model(features, training=False)
    v_loss = loss_object(labels, predictions)
    
    val_loss_metric(v_loss)
    val_acc_metric(labels, predictions)
    val_mae_metric(tf.one_hot(labels, depth=num_classes), predictions)

# 5. Loop Training Eksekusi Epoch
EPOCHS = 30
print("=== MEMULAI PROSES TRAINING (GRADIENT TAPE) ===")

for epoch in range(EPOCHS):
    train_loss_metric.reset_state()
    train_acc_metric.reset_state()
    train_mae_metric.reset_state()
    val_loss_metric.reset_state()
    val_acc_metric.reset_state()
    val_mae_metric.reset_state()

    for x_batch, y_batch in train_dataset:
        train_step(x_batch, y_batch)

    for x_val_batch, y_val_batch in dev_dataset:
        val_step(x_val_batch, y_val_batch)

    # Tulis hasil ke TensorBoard Logs
    with train_summary_writer.as_default():
        tf.summary.scalar('loss', train_loss_metric.result(), step=epoch)
        tf.summary.scalar('accuracy', train_acc_metric.result(), step=epoch)
        tf.summary.scalar('mae', train_mae_metric.result(), step=epoch)
    with val_summary_writer.as_default():
        tf.summary.scalar('loss', val_loss_metric.result(), step=epoch)
        tf.summary.scalar('accuracy', val_acc_metric.result(), step=epoch)
        tf.summary.scalar('mae', val_mae_metric.result(), step=epoch)

    print(f"Epoch {epoch+1:02d}: "
          f"Train Acc: {train_acc_metric.result()*100:.2f}%, Train MAE: {train_mae_metric.result():.4f} | "
          f"Val Acc: {val_acc_metric.result()*100:.2f}%, Val MAE: {val_mae_metric.result():.4f}")

# 6. Ekspor Model Final Siap Produksi
model.save("model_intent_classification_prod.keras")
print("\n=== TRAINING SELESAI & MODEL BERHASIL DIEKSPOR (.keras) ===")

=== MEMULAI PROSES TRAINING (GRADIENT TAPE) ===
Epoch 01: Train Acc: 87.06%, Train MAE: 0.0812 | Val Acc: 92.62%, Val MAE: 0.1036
Epoch 02: Train Acc: 87.99%, Train MAE: 0.0756 | Val Acc: 93.20%, Val MAE: 0.1243
Epoch 03: Train Acc: 87.84%, Train MAE: 0.0757 | Val Acc: 94.23%, Val MAE: 0.0648
Epoch 04: Train Acc: 88.45%, Train MAE: 0.0724 | Val Acc: 94.09%, Val MAE: 0.0468
Epoch 05: Train Acc: 88.74%, Train MAE: 0.0703 | Val Acc: 92.85%, Val MAE: 0.0525
Epoch 06: Train Acc: 89.61%, Train MAE: 0.0672 | Val Acc: 93.54%, Val MAE: 0.0521
Epoch 07: Train Acc: 89.72%, Train MAE: 0.0658 | Val Acc: 93.98%, Val MAE: 0.0479
Epoch 08: Train Acc: 90.30%, Train MAE: 0.0641 | Val Acc: 94.35%, Val MAE: 0.0415
Epoch 09: Train Acc: 91.20%, Train MAE: 0.0606 | Val Acc: 94.18%, Val MAE: 0.0521
Epoch 10: Train Acc: 91.77%, Train MAE: 0.0578 | Val Acc: 94.00%, Val MAE: 0.0487
Epoch 11: Train Acc: 91.78%, Train MAE: 0.0576 | Val Acc: 94.23%, Val MAE: 0.0487
Epoch 12: Train Acc: 91.90%, Train MAE: 0.0567 | V

In [17]:
# Reset metrik untuk mengevaluasi data test terpisah
val_loss_metric.reset_state()
val_acc_metric.reset_state()
val_mae_metric.reset_state()

for x_test_batch, y_test_batch in test_dataset:
    val_step(x_test_batch, y_test_batch)

final_accuracy = val_acc_metric.result().numpy()
final_mae = val_mae_metric.result().numpy()

print("=== HASIL EVALUASI AKHIR MVP ===")
print(f"Akurasi Akhir Model Terhadap Data Test : {final_accuracy*100:.2f}% (Target Minimal: 85%)")
print(f"MAE Akhir Model Terhadap Data Test      : {final_mae:.4f}")

if final_accuracy >= 0.85:
    print("\n🎉 MODEL MEMENUHI SYARAT KELAYAKAN VOICEBANK!")
else:
    print("\n⚠️ MODEL BELUM MEMENUHI SYARAT AKURASI MINIMAL.")

=== HASIL EVALUASI AKHIR MVP ===
Akurasi Akhir Model Terhadap Data Test : 93.93% (Target Minimal: 85%)
MAE Akhir Model Terhadap Data Test      : 0.0460

🎉 MODEL MEMENUHI SYARAT KELAYAKAN VOICEBANK!


In [18]:
print("=== PROSES SIMULASI INFERENCE MODEL ===")

# 1. Load kembali model dari disk untuk memastikan file valid
loaded_model = tf.keras.models.load_model("model_intent_classification_prod.keras")

# 2. Ambil 1 contoh file acak dari X_test untuk simulasi input
sample_index = 10
sample_features = X_test[sample_index]  # Dimensi tunggal: (128, 87)
actual_label = y_test[sample_index]

# 3. Ubah bentuknya ke dimensi batch (1, 128, 87)
input_batch = np.expand_dims(sample_features, axis=0)

# 4. Prediksi Klasifikasi Intent
predictions = loaded_model.predict(input_batch, verbose=0)
predicted_class = np.argmax(predictions[0])
confidence_score = np.max(predictions[0])

# Mapping label teks sederhana
intent_labels = {0: "Transfer Saldo", 1: "Cek Rekening", 2: "Blokir Kartu", 3: "Ubah PIN", 4: "CS"}

print(f"Index Sampel     : {sample_index}")
print(f"Label Asli (ID)  : {actual_label} ({intent_labels.get(actual_label, 'Unknown')})")
print(f"Hasil Prediksi   : {predicted_class} ({intent_labels.get(predicted_class, 'Unknown')})")
print(f"Confidence Score : {confidence_score * 100:.2f}%")

=== PROSES SIMULASI INFERENCE MODEL ===
Index Sampel     : 10
Label Asli (ID)  : 0 (Transfer Saldo)
Hasil Prediksi   : 0 (Transfer Saldo)
Confidence Score : 93.70%
